In [1]:
# ---- Setup: Drive + code + wandb (run this first, every session) ----
REPO_URL = "https://github.com/ammara-asif/fyp.git"   # <-- EDIT ONCE
BRANCH = "nested-b-d"   # the branch with the code this notebook needs (set back to "main" after merging)

import os, subprocess, sys
from google.colab import drive, userdata

# Drive must be mounted from the notebook kernel (never from a !python subprocess).
drive.mount("/content/drive")

# Code: fresh clone each session (Colab disk is wiped on reset), pull if it already exists.
# For a private repo, add a Colab secret GITHUB_TOKEN (a fine-grained read-only token).
CODE = "/content/fyp"
url = REPO_URL
try:
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "-q", url, CODE], check=True)
# fetch + checkout + pull, so a clone left on another branch earlier in the session switches over
subprocess.run(["git", "-C", CODE, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", CODE, "pull", "-q", "origin", BRANCH], check=True)
print(BRANCH, subprocess.check_output(["git", "-C", CODE, "log", "-1", "--oneline"]).decode())

# Dependencies (torch/torchvision come preinstalled with Colab's CUDA build; don't reinstall them).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

# wandb: key comes from the Colab secret WANDB_API_KEY (key icon in the left sidebar). Never hardcode it.
import wandb
wandb.login(key=userdata.get("WANDB_API_KEY"))

sys.path.insert(0, f"{CODE}/src")
# Python caches imported modules for the whole kernel session, so after a git pull
# a plain `import` would silently keep running the OLD code. Drop the cached copies.
_MINE = {"fyp_infra", "stage1_runner", "stage1_analysis", "stage1_drift_diagnostic",
         "stage1_structural", "nested_builder"}
for _m in list(sys.modules):
    if _m in _MINE or _m == "src" or _m.startswith("src."):   # src.* = the team's nested code
        sys.modules.pop(_m, None)
import torch
from fyp_infra import setup_paths, run_sweep, run_id, load_result
import stage1_runner as sr
paths = setup_paths()   # -> /content/drive/MyDrive/fyp_nested_cl/{data,results,logs}
print(paths)
# Hard stop for training: a CPU run is ~12x slower and its timings are useless for budgeting.
REQUIRE_GPU = True
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)")

Mounted at /content/drive
nested-b-d 9a30690 updated



/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ammara-asif125 (kashaf-saleem09-institute-of-business-administration) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


{'root': '/content/drive/MyDrive/fyp_nested_cl', 'data': '/content/drive/MyDrive/fyp_nested_cl/data', 'results': '/content/drive/MyDrive/fyp_nested_cl/results', 'logs': '/content/drive/MyDrive/fyp_nested_cl/logs', 'checkpoints': '/content/drive/MyDrive/fyp_nested_cl/checkpoints'}
GPU: Tesla T4


In [2]:
import numpy as np, pandas as pd

def table(configs):
    """One row per config: accuracy metrics, cost, and per-level means over tasks (nested arms)."""
    rows = []
    for cfg in configs:
        base = {"arm": sr.arm_label(cfg), "lr": cfg["lr"], "seed": cfg["seed"]}
        r = load_result(cfg, paths)
        if r is None:
            rows.append({**base, "status": "not finished"}); continue
        m = r["metrics"]
        row = {**base, "run_id": r["run_id"], "avg_incremental_acc": m["avg_incremental_acc"],
               "final_avg_acc": m["final_avg_acc"], "avg_forgetting": m["avg_forgetting"],
               # mean of the diagonal: accuracy on each task right after learning it (plasticity check)
               "acc_just_learned": float(np.mean(np.diag(m["acc_matrix"]))),
               "steps": m["optimizer_steps"], "n_params": m["n_params"],
               "minutes": r["wall_clock_s"] / 60, "peak_gpu_mb": m.get("peak_gpu_mem_mb")}
        log = m.get("level_log_per_exp") or []
        for k in (log[0] if log else {}):
            if k.startswith(("updates/", "rel_displacement/", "deployed_rel_displacement/", "penalty_ratio/")):
                row[k] = float(np.mean([x[k] for x in log]))
        rows.append(row)
    return pd.DataFrame(rows)

In [4]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q"], cwd=CODE, capture_output=True, text=True,
                   env={**os.environ, "WANDB_MODE": "disabled", "CUDA_VISIBLE_DEVICES": ""})   # CPU: deterministic
print(r.stdout[-6000:]); print(r.stderr[-3000:])
assert r.returncode == 0, "tests failed: read the output above before running anything else"

.....................                                                    [100%]
=============================== warnings summary ===============================
tests/test_nested.py: 10 warnings
tests/test_smoke.py: 5 warnings
  /usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
    warnings.warn("No loggers specified, metrics will not be logged")

tests/test_nested.py::test_flat_nested_path_equals_naive
tests/test_nested.py::test_b_per_level_lrs
tests/test_nested.py::test_d_runs_steps_and_checkpoints
tests/test_nested.py::test_a_update_counts
tests/test_nested.py::test_a_matches_d_with_zero_penalty
tests/test_nested.py::test_a_matches_d_with_zero_penalty
tests/test_nested.py::test_c_with_zero_taus_equals_flat
tests/test_nested.py::test_c_with_zero_taus_equals_flat
tests/test_nested.py::test_c_runs_and_checkpoints_shadow
  /usr/local/lib/python3.13/dist-packages/avalanche/training/templates/

In [14]:
LONG_TAUS = [0.0, 0.99, 0.999]   # fastest first: stem+layer1 averages over ~1000 steps, layer2+layer3 over ~100
SMOKE = [sr.arm_cfg("nested_a", stage="smoke_nested", n_experiences=2, epochs=1, lr=0.1, eval_split="val"),
         sr.arm_cfg("nested_c", stage="smoke_nested", n_experiences=2, epochs=1, lr=0.1, eval_split="val"),
         sr.arm_cfg("nested_c", stage="smoke_nested", n_experiences=2, epochs=1, lr=0.1, eval_split="val",
                    nesting=dict(ema_taus=LONG_TAUS))]
out = run_sweep(SMOKE, sr.run_experiment, paths, force=True)   # force: always re-time after code changes
smoke = table(SMOKE)
smoke["train_min"] = [sum(out[run_id(c)]["metrics"]["train_minutes_per_exp"]) if out[run_id(c)] else None
                      for c in SMOKE]
smoke["est_full_run_train_min"] = 10 * smoke["train_min"]
display(smoke.T)

[1/3] cee3d824aa16 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/2: seen-avg 19.64%  current 19.64%  train-loss 3.510
  exp 2/2: seen-avg 9.88%  current 19.76%  train-loss 3.520


acc_current_exp,▁█
acc_exp/00,█▁
acc_exp/01,▁
acc_seen_avg,█▁
exp,▁█
level/displacement/layer2+layer3,█▁
level/displacement/layer4+fc,▁█
level/displacement/stem+layer1,█▁
level/rel_displacement/layer2+layer3,█▁
level/rel_displacement/layer4+fc,▁█
+7,...


[2/3] ab1361594182 running


...

[3/3] fc98dd5d06f4 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/2: seen-avg 15.00%  current 15.00%  train-loss 3.418
  exp 2/2: seen-avg 6.94%  current 13.88%  train-loss 3.422


acc_current_exp,█▁
acc_exp/00,█▁
acc_exp/01,▁
acc_seen_avg,█▁
exp,▁█
level/deployed_displacement/layer2+layer3,█▁
level/deployed_displacement/layer4+fc,▁█
level/deployed_displacement/stem+layer1,▁█
level/deployed_rel_displacement/layer2+layer3,█▁
level/deployed_rel_displacement/layer4+fc,▁█
+13,...


,0,1,2
arm,nested_a_ep1,nested_c_cw0.1_ep1,nested_c_cw0.1_tau0-0.99-0.999_ep1
lr,0.1,0.1,0.1
seed,0,0,0
run_id,cee3d824aa16,ab1361594182,fc98dd5d06f4
avg_incremental_acc,14.76,16.26,10.97
final_avg_acc,9.88,11.32,6.94
avg_forgetting,19.64,21.2,15.0
acc_just_learned,19.7,21.92,14.44
steps,1408,1408,1408
n_params,1109240,1109240,1109240


In [5]:
EPOCHS = 10

def val_cfg(name, lr, **kw):
    return sr.arm_cfg(name, stage="stage1_nested_lr_grid", epochs=EPOCHS, seed=0, lr=lr, eval_split="val", **kw)

D_ZERO = val_cfg("nested_d", 0.7, rule_kwargs=dict(surprise_weight=0.0))   # from stage1_nested_bd, reused
A_REG = val_cfg("nested_a", 0.7)
print(run_id(D_ZERO), sr.arm_label(D_ZERO), "| exists:", load_result(D_ZERO, paths) is not None)
out = run_sweep([A_REG], sr.run_experiment, paths)
reg = table([D_ZERO, A_REG])
display(reg[["arm", "lr", "run_id", "avg_incremental_acc", "acc_just_learned", "final_avg_acc", "minutes"]])
if reg["avg_incremental_acc"].notna().all():
    d = abs(reg["avg_incremental_acc"].iloc[0] - reg["avg_incremental_acc"].iloc[1])
    print(f"|diff| avg incremental {d:.2f}")
    print("REGRESSION CHECK", "PASSED" if d < 0.5 else "FAILED: stop and investigate")

4946950b08e0 nested_d_lam0 | exists: True
[1/1] 77e98226ff0a already finished, skipping


,arm,lr,run_id,avg_incremental_acc,acc_just_learned,final_avg_acc,minutes
0,nested_d_lam0,0.7,4946950b08e0,14.179952,82.62,4.48,18.304550
1,nested_a,0.7,77e98226ff0a,14.382958,82.28,4.38,7.896123


|diff| avg incremental 0.20
REGRESSION CHECK PASSED


In [9]:
A_GRID = [0.1, 0.3, 0.5, 0.7, 0.9]
A_CONFIGS = [val_cfg("nested_a", lr) for lr in A_GRID]
A_WIDE_CONFIGS = [val_cfg("nested_a", lr, nesting=dict(periods=[1, 16, 256])) for lr in A_GRID]
for c in A_CONFIGS + A_WIDE_CONFIGS: print(run_id(c), sr.arm_label(c), c["lr"])
out = run_sweep(A_CONFIGS + A_WIDE_CONFIGS, sr.run_experiment, paths)

dd6bd4658e58 nested_a 0.1
4c5a2522c5b9 nested_a 0.3
fa247b89e02b nested_a 0.5
77e98226ff0a nested_a 0.7
39122210a912 nested_a 0.9
c5e48bba7a72 nested_a_1-16-256 0.1
210533da4841 nested_a_1-16-256 0.3
eacea3855c4a nested_a_1-16-256 0.5
d3be66f70c4a nested_a_1-16-256 0.7
50aceeef8da3 nested_a_1-16-256 0.9
[1/10] dd6bd4658e58 already finished, skipping
[2/10] 4c5a2522c5b9 already finished, skipping
[3/10] fa247b89e02b already finished, skipping
[4/10] 77e98226ff0a already finished, skipping
[5/10] 39122210a912 already finished, skipping
[6/10] c5e48bba7a72 already finished, skipping
[7/10] 210533da4841 already finished, skipping
[8/10] eacea3855c4a already finished, skipping
[9/10] d3be66f70c4a already finished, skipping
[10/10] 50aceeef8da3 already finished, skipping


In [10]:
def pick(configs, grid, name):
    t = table(configs)
    display(t.drop(columns=["seed", "run_id"], errors="ignore"))
    done = t.dropna(subset=["avg_incremental_acc"]) if "avg_incremental_acc" in t else t.iloc[:0]
    if len(done):
        best = done.loc[done["avg_incremental_acc"].idxmax()]
        print(f"best {name} lr: {best['lr']} (avg incremental {best['avg_incremental_acc']:.2f})")
        if best["lr"] in (min(grid), max(grid)): print(f"WARNING: best {name} lr is at the edge of its grid; extend it first.")

pick(A_CONFIGS, A_GRID, "A (1,4,16)")
pick(A_WIDE_CONFIGS, A_GRID, "A (1,16,256)")

,arm,lr,avg_incremental_acc,final_avg_acc,avg_forgetting,acc_just_learned,steps,n_params,minutes,peak_gpu_mb,updates/stem+layer1,rel_displacement/stem+layer1,updates/layer2+layer3,rel_displacement/layer2+layer3,updates/layer4+fc,rel_displacement/layer4+fc
0,nested_a,0.1,13.527405,4.24,75.978947,76.42,14200,1109240,7.996267,215.729980,44.0,0.013209,177.0,0.019201,710.0,0.072262
1,nested_a,0.3,14.249783,4.26,81.684211,81.86,14200,1109240,7.966201,221.063965,44.0,0.029982,177.0,0.046504,710.0,0.109357
2,nested_a,0.5,14.410670,4.54,82.189474,82.62,14200,1109240,7.954976,202.917480,44.0,0.047179,177.0,0.072008,710.0,0.151360
3,nested_a,0.7,14.382958,4.38,82.000000,82.28,14200,1109240,7.896123,210.395996,44.0,0.066911,177.0,0.093832,710.0,0.196261
4,nested_a,0.9,14.318991,4.50,82.989474,83.34,14200,1109240,7.854058,208.251465,44.0,0.082393,177.0,0.118036,710.0,0.230999


best A (1,4,16) lr: 0.5 (avg incremental 14.41)


,arm,lr,avg_incremental_acc,final_avg_acc,avg_forgetting,acc_just_learned,steps,n_params,minutes,peak_gpu_mb,updates/stem+layer1,rel_displacement/stem+layer1,updates/layer2+layer3,rel_displacement/layer2+layer3,updates/layer4+fc,rel_displacement/layer4+fc
0,nested_a_1-16-256,0.1,12.212955,3.76,69.368421,69.66,14200,1109240,7.911083,213.585449,2.0,0.001149,44.0,0.006075,710.0,0.074287
1,nested_a_1-16-256,0.3,13.124949,3.98,74.842105,75.08,14200,1109240,7.900830,202.917480,2.0,0.002594,44.0,0.013713,710.0,0.102869
2,nested_a_1-16-256,0.5,13.404031,4.10,76.842105,77.10,14200,1109240,7.752197,208.251465,2.0,0.003916,44.0,0.021034,710.0,0.128993
3,nested_a_1-16-256,0.7,13.645873,4.30,78.021053,78.42,14200,1109240,7.737347,202.917480,2.0,0.005430,44.0,0.027663,710.0,0.151985
4,nested_a_1-16-256,0.9,13.866282,4.32,78.652632,79.04,14200,1109240,7.821627,208.251465,2.0,0.006910,44.0,0.034445,710.0,0.179472


best A (1,16,256) lr: 0.9 (avg incremental 13.87)


In [11]:
C_GRID = [0.03, 0.1, 0.2, 0.3]
C_CONFIGS = [val_cfg("nested_c", lr) for lr in C_GRID]
for c in C_CONFIGS: print(run_id(c), sr.arm_label(c), c["lr"])
out = run_sweep(C_CONFIGS, sr.run_experiment, paths)
pick(C_CONFIGS, C_GRID, "C")

0cebbd99b8d4 nested_c_cw0.1 0.03
7d4f76d61224 nested_c_cw0.1 0.1
61d70cdcaa93 nested_c_cw0.1 0.2
44cb3d155e9e nested_c_cw0.1 0.3
[1/4] 0cebbd99b8d4 already finished, skipping
[2/4] 7d4f76d61224 already finished, skipping
[3/4] 61d70cdcaa93 already finished, skipping
[4/4] 44cb3d155e9e already finished, skipping


,arm,lr,avg_incremental_acc,final_avg_acc,avg_forgetting,acc_just_learned,steps,n_params,minutes,peak_gpu_mb,updates/stem+layer1,rel_displacement/stem+layer1,updates/layer2+layer3,rel_displacement/layer2+layer3,updates/layer4+fc,rel_displacement/layer4+fc,deployed_rel_displacement/stem+layer1,deployed_rel_displacement/layer2+layer3,deployed_rel_displacement/layer4+fc
0,nested_c_cw0.1,0.03,13.929522,4.08,79.368421,79.48,14200,1109240,9.412329,208.117188,710.0,0.048850,710.0,0.024779,710.0,0.053687,0.048265,0.024733,0.053687
1,nested_c_cw0.1,0.10,15.049345,4.26,85.305263,85.30,14200,1109240,8.863981,220.929688,710.0,0.098692,710.0,0.063894,710.0,0.079332,0.097373,0.063720,0.079332
2,nested_c_cw0.1,0.20,14.863430,4.52,85.242105,85.50,14200,1109240,9.231794,233.742188,710.0,0.148334,710.0,0.112631,710.0,0.128842,0.146229,0.112289,0.128842
3,nested_c_cw0.1,0.30,13.924705,4.18,81.221053,81.34,14200,1109240,9.320236,220.929688,710.0,0.193402,710.0,0.163092,710.0,0.192062,0.191425,0.162788,0.192062


best C lr: 0.1 (avg incremental 15.05)


In [12]:
C_LR = 0.1                       # <- fill in from the table above
CW_GRID = [0.0, 0.1, 1.0]         # 0.1 at C_LR is already in the lr grid (same config -> reused)
if C_LR is None:
    print("set C_LR first")
else:
    CW_CONFIGS = [val_cfg("nested_c", C_LR, rule_kwargs=dict(consistency_weight=w)) for w in CW_GRID]
    for c in CW_CONFIGS: print(run_id(c), sr.arm_label(c), c["lr"])
    out = run_sweep(CW_CONFIGS, sr.run_experiment, paths)
    display(table(CW_CONFIGS).drop(columns=["seed", "run_id"], errors="ignore"))

3107ff608870 nested_c_cw0 0.1
7d4f76d61224 nested_c_cw0.1 0.1
2eaedc36fdc2 nested_c_cw1 0.1
[1/3] 3107ff608870 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 72.80%  current 72.80%  train-loss 0.923
  exp 2/20: seen-avg 44.60%  current 89.20%  train-loss 0.580
  exp 3/20: seen-avg 26.80%  current 80.40%  train-loss 0.887
  exp 4/20: seen-avg 22.10%  current 88.40%  train-loss 0.583
  exp 5/20: seen-avg 16.16%  current 80.80%  train-loss 0.893
  exp 6/20: seen-avg 13.93%  current 83.60%  train-loss 0.808
  exp 7/20: seen-avg 12.17%  current 85.20%  train-loss 0.678
  exp 8/20: seen-avg 11.05%  current 88.40%  train-loss 0.520
  exp 9/20: seen-avg 9.20%  current 82.80%  train-loss 0.683
  exp 10/20: seen-avg 8.64%  current 86.40%  train-loss 0.614
  exp 11/20: seen-avg 7.35%  current 80.80%  train-loss 0.633
  exp 12/20: seen-avg 6.13%  current 73.60%  train-loss 0.737
  exp 13/20: seen-avg 6.34%  current 82.40%  train-loss 0.849
  exp 14/20: seen-avg 6.29%  current 88.00%  train-loss 0.546
  exp 15/20: seen-avg 5.63%  current 84.40%  train-loss 0.684
  exp 16/20: seen-avg 5.62%  current 90.00%  train-loss 0.566
  exp 17/

acc_current_exp,▁▇▄▇▄▅▆▇▅▆▄▁▅▇▅█▇▇▅█
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+31,...


[2/3] 7d4f76d61224 already finished, skipping
[3/3] 2eaedc36fdc2 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 69.60%  current 69.60%  train-loss 0.993
  exp 2/20: seen-avg 44.20%  current 88.40%  train-loss 12.388
  exp 3/20: seen-avg 21.47%  current 64.40%  train-loss 20.061
  exp 4/20: seen-avg 19.90%  current 79.60%  train-loss 22.816
  exp 5/20: seen-avg 14.16%  current 70.80%  train-loss 48.222
  exp 6/20: seen-avg 9.93%  current 59.60%  train-loss 42.662
  exp 7/20: seen-avg 4.51%  current 30.80%  train-loss 84.328
  exp 8/20: seen-avg 8.55%  current 68.40%  train-loss 31.413
  exp 9/20: seen-avg 8.04%  current 72.40%  train-loss 3.884
  exp 10/20: seen-avg 7.00%  current 68.40%  train-loss 4.382
  exp 11/20: seen-avg 4.98%  current 54.80%  train-loss 27.954
  exp 12/20: seen-avg 5.63%  current 67.20%  train-loss 4.975
  exp 13/20: seen-avg 2.95%  current 38.00%  train-loss 27.328
  exp 14/20: seen-avg 4.91%  current 68.80%  train-loss 4.095
  exp 15/20: seen-avg 4.21%  current 63.20%  train-loss 7.549
  exp 16/20: seen-avg 4.47%  current 71.60%  train-loss 10.659
  

acc_current_exp,▆█▅▇▆▄▁▆▆▆▄▅▂▆▅▆▇▅▅▇
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+31,...


,arm,lr,avg_incremental_acc,final_avg_acc,avg_forgetting,acc_just_learned,steps,n_params,minutes,peak_gpu_mb,updates/stem+layer1,rel_displacement/stem+layer1,updates/layer2+layer3,rel_displacement/layer2+layer3,updates/layer4+fc,rel_displacement/layer4+fc,deployed_rel_displacement/stem+layer1,deployed_rel_displacement/layer2+layer3,deployed_rel_displacement/layer4+fc
0,nested_c_cw0,0.1,14.696306,4.56,84.126316,84.48,14200,1109240,8.487420,233.742188,710.0,0.097497,710.0,0.063163,710.0,0.078700,0.096329,0.063007,0.078700
1,nested_c_cw0.1,0.1,15.049345,4.26,85.305263,85.30,14200,1109240,8.863981,220.929688,710.0,0.098692,710.0,0.063894,710.0,0.079332,0.097373,0.063720,0.079332
2,nested_c_cw1,0.1,12.543878,4.10,65.747368,66.56,14200,1109240,9.390045,246.554688,710.0,0.088053,710.0,0.052031,710.0,0.120363,0.086951,0.051862,0.120363


In [15]:
C_CW = 0.1                  # <- fill in from the table above (highest avg_incremental_acc)
if C_LR is None or C_CW is None:
    print("set C_LR and C_CW first")
else:
    pair = [val_cfg("nested_c", C_LR, rule_kwargs=dict(consistency_weight=C_CW)),
            val_cfg("nested_c", C_LR, rule_kwargs=dict(consistency_weight=C_CW), nesting=dict(ema_taus=LONG_TAUS))]
    out = run_sweep(pair, sr.run_experiment, paths)
    t = table(pair)
    display(t.drop(columns=["seed", "run_id"], errors="ignore").T)
    if t["avg_incremental_acc"].notna().all():
        gain = t["avg_incremental_acc"].iloc[1] - t["avg_incremental_acc"].iloc[0]
        print(f"long tau minus teammate's tau: {gain:+.2f} points avg incremental")
        print("-> take the long tau to the supervisor" if gain > 0.5 else "-> keep the teammate's tau for the control")

[1/2] 7d4f76d61224 already finished, skipping
[2/2] 4a47ed75f45f running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 50.40%  current 50.40%  train-loss 0.955
  exp 2/20: seen-avg 39.40%  current 78.80%  train-loss 0.600
  exp 3/20: seen-avg 13.73%  current 41.20%  train-loss 0.929
  exp 4/20: seen-avg 19.00%  current 76.00%  train-loss 0.584
  exp 5/20: seen-avg 14.88%  current 74.40%  train-loss 0.935
  exp 6/20: seen-avg 10.40%  current 62.40%  train-loss 0.928
  exp 7/20: seen-avg 11.03%  current 77.20%  train-loss 0.718
  exp 8/20: seen-avg 10.30%  current 82.40%  train-loss 0.553
  exp 9/20: seen-avg 8.93%  current 80.40%  train-loss 0.672
  exp 10/20: seen-avg 8.52%  current 85.20%  train-loss 0.650
  exp 11/20: seen-avg 7.16%  current 78.80%  train-loss 0.620
  exp 12/20: seen-avg 6.07%  current 72.80%  train-loss 0.780
  exp 13/20: seen-avg 5.85%  current 76.00%  train-loss 0.866
  exp 14/20: seen-avg 6.20%  current 86.80%  train-loss 0.614
  exp 15/20: seen-avg 5.33%  current 80.00%  train-loss 0.691
  exp 16/20: seen-avg 5.00%  current 80.00%  train-loss 0.576
  exp 17/

acc_current_exp,▂▇▁▆▆▄▆▇▇█▇▆▆█▇▇▇▆▅█
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+31,...


,0,1
arm,nested_c_cw0.1,nested_c_cw0.1_tau0-0.99-0.999
lr,0.1,0.1
avg_incremental_acc,15.049345,11.949775
final_avg_acc,4.26,4.36
avg_forgetting,85.305263,73.978947
acc_just_learned,85.3,74.64
steps,14200,14200
n_params,1109240,1109240
minutes,8.863981,9.310342
peak_gpu_mb,220.929688,233.742188


long tau minus teammate's tau: -3.10 points avg incremental
-> keep the teammate's tau for the control


In [19]:
SEEDS = [0, 1, 2]
LR = {"nested_a": 0.5, "nested_c": 0.1}
C_CW_CONTROL = 0.1                # usually = C_CW
CONTROL_AC = []
for s in SEEDS:
    for name, lr in LR.items():
        if lr is None or (name == "nested_c" and C_CW_CONTROL is None):
            continue
        kw = dict(rule_kwargs=dict(consistency_weight=C_CW_CONTROL)) if name == "nested_c" else {}
        CONTROL_AC.append(sr.arm_cfg(name, stage="stage1_nested_control", epochs=EPOCHS, seed=s, lr=lr,
                                     eval_split="test", save_checkpoints=True, **kw))
for name, lr in LR.items():
    if lr is None or (name == "nested_c" and C_CW_CONTROL is None): print(f"skipping {name}: set LR / C_CW_CONTROL first")
for c in CONTROL_AC: print(run_id(c), sr.arm_label(c), c["lr"], c["seed"])

82f66588c5f1 nested_a 0.5 0
f2ad0b0fb4ea nested_c_cw0.1 0.1 0
1a6e0b7da780 nested_a 0.5 1
ca8ab2efedc2 nested_c_cw0.1 0.1 1
9da44b28d332 nested_a 0.5 2
82d25ec4d308 nested_c_cw0.1 0.1 2


In [20]:
out = run_sweep(CONTROL_AC, sr.run_experiment, paths)

[1/6] 82f66588c5f1 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 71.40%  current 71.40%  train-loss 0.915
  exp 2/20: seen-avg 42.30%  current 84.60%  train-loss 0.501
  exp 3/20: seen-avg 22.53%  current 67.60%  train-loss 0.849
  exp 4/20: seen-avg 22.00%  current 88.00%  train-loss 0.484
  exp 5/20: seen-avg 15.12%  current 75.60%  train-loss 0.788
  exp 6/20: seen-avg 13.53%  current 81.20%  train-loss 0.716
  exp 7/20: seen-avg 12.00%  current 84.00%  train-loss 0.591
  exp 8/20: seen-avg 10.70%  current 85.60%  train-loss 0.454
  exp 9/20: seen-avg 8.84%  current 79.60%  train-loss 0.550
  exp 10/20: seen-avg 8.48%  current 84.80%  train-loss 0.538
  exp 11/20: seen-avg 7.98%  current 87.80%  train-loss 0.543
  exp 12/20: seen-avg 6.58%  current 79.00%  train-loss 0.657
  exp 13/20: seen-avg 5.94%  current 77.20%  train-loss 0.715
  exp 14/20: seen-avg 6.39%  current 89.40%  train-loss 0.451
  exp 15/20: seen-avg 5.65%  current 84.80%  train-loss 0.624
  exp 16/20: seen-avg 5.49%  current 87.80%  train-loss 0.485
  exp 17/

acc_current_exp,▂▆▁▇▃▅▆▆▅▆▇▄▄▇▆▇██▅█
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+25,...


[2/6] f2ad0b0fb4ea running


...

acc_current_exp,▂▆▇▁▅▆▇▇█▆▇█▃▇▆▆▃▇▇▅
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+25,...


[6/6] 82d25ec4d308 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 82.00%  current 82.00%  train-loss 0.834
  exp 2/20: seen-avg 42.90%  current 85.80%  train-loss 0.677
  exp 3/20: seen-avg 28.80%  current 86.40%  train-loss 0.675
  exp 4/20: seen-avg 20.25%  current 81.00%  train-loss 0.787
  exp 5/20: seen-avg 16.92%  current 84.60%  train-loss 0.615
  exp 6/20: seen-avg 14.70%  current 88.20%  train-loss 0.671
  exp 7/20: seen-avg 12.69%  current 88.80%  train-loss 0.601
  exp 8/20: seen-avg 10.95%  current 87.60%  train-loss 0.629
  exp 9/20: seen-avg 10.42%  current 93.80%  train-loss 0.481
  exp 10/20: seen-avg 8.48%  current 84.80%  train-loss 0.771
  exp 11/20: seen-avg 8.11%  current 89.20%  train-loss 0.610
  exp 12/20: seen-avg 7.68%  current 92.20%  train-loss 0.478
  exp 13/20: seen-avg 5.68%  current 73.80%  train-loss 0.848
  exp 14/20: seen-avg 6.47%  current 90.60%  train-loss 0.563
  exp 15/20: seen-avg 5.61%  current 84.20%  train-loss 0.643
  exp 16/20: seen-avg 5.41%  current 86.60%  train-loss 0.622
  exp 17

acc_current_exp,▄▅▆▄▅▆▆▆█▅▇▇▂▇▅▆▁▆▇▄
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+31,...


In [21]:
# Flat, B and D control runs from stage1_nested_bd (reused, not re-run), for one table
BD = [sr.arm_cfg(name, stage="stage1_nested_control", epochs=EPOCHS, seed=s, lr=lr, eval_split="test",
                 save_checkpoints=True, **kw)
      for s in SEEDS for name, lr, kw in [("flat", 0.1, {}), ("nested_b", 0.6, {}),
                                          ("nested_d", 0.7, dict(rule_kwargs=dict(surprise_weight=0.01)))]]
cdf = table(BD + CONTROL_AC)
done = cdf[cdf["status"] != "not finished"] if "status" in cdf else cdf
assert done["steps"].nunique() == 1 and done["n_params"].nunique() == 1, "steps or param count differ between arms. The comparison is NOT matched"
cols = ["avg_incremental_acc", "acc_just_learned", "final_avg_acc", "avg_forgetting", "minutes", "peak_gpu_mb"]
display(done.groupby(["arm", "lr"])[cols].agg(["mean", "std", "count"]).round(2))
lvl = [c for c in done.columns if c.startswith(("updates/", "rel_displacement/", "deployed_rel_displacement/", "penalty_ratio/"))]
if lvl: display(done.groupby(["arm", "lr"])[lvl].mean().T.round(4))

avg_incremental_acc             acc_just_learned        \
                                    mean   std count             mean   std   
arm              lr                                                           
flat_nl          0.1               14.19  0.38     3            82.15  0.32   
nested_a         0.5               14.06  0.33     3            83.16  0.07   
nested_b         0.6               14.17  0.49     3            81.93  0.77   
nested_c_cw0.1   0.1               14.99  0.29     3            85.95  0.36   
nested_d_lam0.01 0.7               14.01  0.39     3            83.12  0.41   

                           final_avg_acc             avg_forgetting        \
                     count          mean   std count           mean   std   
arm              lr                                                         
flat_nl          0.1     3          4.19  0.26     3          82.06  0.45   
nested_a         0.5     3          4.35  0.22     3          82.96  0.23   
nested_b         0.6     3          4.21  0.37     3          81.81  1.18   
nested_c_cw0.1   0.1     3          4.35  0.30     3          85.89  0.11   
nested_d_lam0.01 0.7     3          4.29  0.19     3          82.98  0.33   

                           minutes             peak_gpu_mb               
                     count    mean   std count        mean    std count  
arm              lr                                                      
flat_nl          0.1     3    8.79  0.07     3      211.90   6.16     3  
nested_a         0.5     3    8.68  0.04     3      223.35  11.72     3  
nested_b         0.6     3    8.65  0.06     3      215.09   3.08     3  
nested_c_cw0.1   0.1     3   10.27  0.04     3      221.96  10.48     3  
nested_d_lam0.01 0.7     3   19.98  0.08     3      220.43   3.08     3

arm,flat_nl,nested_a,nested_b,nested_c_cw0.1,nested_d_lam0.01
lr,0.1,0.5,0.6,0.1,0.7
updates/stem+layer1,790.0000,49.0000,790.0000,790.0000,49.0000
rel_displacement/stem+layer1,0.0993,0.0496,0.0577,0.0994,0.0739
updates/layer2+layer3,790.0000,197.0000,790.0000,790.0000,197.0000
rel_displacement/layer2+layer3,0.0673,0.0743,0.0821,0.0672,0.0944
updates/layer4+fc,790.0000,790.0000,790.0000,790.0000,790.0000
rel_displacement/layer4+fc,0.0806,0.1592,0.1704,0.0826,0.1979
penalty_ratio/stem+layer1,NaN,NaN,NaN,NaN,0.4448
penalty_ratio/layer2+layer3,NaN,NaN,NaN,NaN,0.0762
deployed_rel_displacement/stem+layer1,NaN,NaN,NaN,0.0983,NaN
